In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [15]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool
from langchain.agents import create_agent




In [3]:
loader = PyPDFLoader("../utils/GenAI_Interview_Questions_Freshers_2026.pdf")
docs = loader.load()

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_docs = splitter.split_documents(docs)

In [7]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

In [8]:
vector_store = InMemoryVectorStore.from_documents(
    documents=splitted_docs,
    embedding=embeddings
)

In [20]:
# Agent = Tools + LLM + Prompt

@tool
def retriever_tool(query: str):
    """
    Retrieve relevant information from the PDF documents.

    The PDF documents contain frequently asked Generative AI
    interview questions and answers for freshers.
    """
    print(f"Tool Used for: {query}")
    
    docs = vector_store.similarity_search(query, k=4)

    if not docs:
        return "No relevant information was found in the documents."

    context = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    return context

In [26]:
llm = ChatOpenAI(
    model="openai/gpt-oss-20b:free",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

In [42]:
system_prompt = """
You are a Generative AI interview assistant.

You have access to retriever_tool, which searches the
provided PDF documents.

IMPORTANT:

For every distinct question or sub-question in the user's
message, call retriever_tool separately.

If the user asks two questions, make two separate tool calls.

For example, if the user asks:

"What is prompt engineering and what is the difference
between GPT, BERT, and T5?"

You should call:

retriever_tool("What is prompt engineering?")

and then:

retriever_tool("What is the difference between GPT, BERT, and T5?")

Do not combine multiple distinct questions into one retrieval query.

Use the retrieved information to formulate the final answer.
"""

In [43]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=system_prompt
)

In [44]:
query = "what is prompt Engineering, and What is the difference between GPT, BERT, and T5?"
res = agent.invoke(
    {"messages": [{"role": "user", "content":query}]}
)

Tool Used for: What is prompt Engineering?
Tool Used for: difference between GPT BERT and T5


In [45]:
result = res["messages"][-1].content

In [46]:
print(result)

**1. What is Prompt Engineering?**  
Prompt engineering is the art of designing and structuring the text you feed to a large language model (LLM) so that the model produces the most accurate, useful, and relevant output possible.  
Key points from the PDF:

| Component | Purpose | Example |
|-----------|---------|---------|
| **System prompt** | Sets the role and constraints (e.g., “You are a helpful customer‑support agent.”) | “You are a friendly tutor.” |
| **Instructions** | Tells the model what to do (e.g., “Summarize this document in 3 bullet points.”) | “Explain the concept in simple terms.” |
| **Context** | Provides background information that the model can use. | A short excerpt from a document. |
| **Examples** | Shows the model the desired style or format. | “Example: 1. … 2. …” |
| **Output format** | Specifies the structure of the answer (JSON, list, table, etc.). | “Return a JSON array.” |

Why it matters  
* The same LLM can give wildly different results depending on how